# 02 — Preparação dos Dados

Grupo 1 — Breno Luna e Paula Carlesso

Roda o pipeline de `src/etl/` e confere o resultado. A regra que governa esta
etapa: **nenhuma feature pode usar informação posterior à transação**.

In [ ]:
import sys
sys.path.append("..")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import ARQUIVO_DATASET, COLUNA_ALVO, FIGURES_DIR

pd.set_option("display.max_columns", 60)
sns.set_theme(style="whitegrid")

In [ ]:
from src.etl.run_pipeline import main as rodar_pipeline

rodar_pipeline()

## Conferindo o dataset gerado

In [ ]:
df = pd.read_parquet(ARQUIVO_DATASET)
print(df.shape)
df.head()

## Validações

In [ ]:
from src.features.build_features import COLUNAS_MODELO

assert df[COLUNA_ALVO].isin([0, 1]).all(), "alvo com valores inesperados"
assert not df["id_transacao"].duplicated().any(), "id_transacao duplicado passou"

faltando = [c for c in COLUNAS_MODELO if c not in df.columns]
print("Features ausentes:", faltando or "nenhuma")

# A primeira transação de cada cartão não pode ter média histórica
primeiras = df.sort_values("data_hora").groupby("id_cartao").head(1)
print("Primeiras transações com média preenchida (deve ser 0):",
      primeiras["media_valor_cartao_ate_agora"].notna().sum())

Esse último teste é o guarda-costas contra vazamento: se a primeira
transação de um cartão já tiver média histórica, é porque o `shift(1)` sumiu e
a feature está olhando o futuro.